# Experiment 5: wide no-PCA Sobol metric landscape

This experiment holds the original second-order PCC representation and Leiden resolution fixed while exploring `k_intra`, `k_inter`, `cutoff_intra`, and `cutoff_inter`. It generates 512 scrambled Sobol settings, processes them in balanced 128-run stages, and runs cheaper predicted graphs first within each stage. Every run is checkpointed and its graph and evaluation tables are released before the next run.

In [ ]:
from pathlib import Path
import sys

def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")

PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")

In [ ]:
import anndata as ad
import pandas as pd

from cross_species import (
    DEFAULT_PARAMETER_RANGES, collect_search_results, load_core_run,
    plot_metric_heatmap, run_wide_sobol_search,
)

PARAMETER_RANGES = dict(DEFAULT_PARAMETER_RANGES)
N_SETTINGS = 512
STAGE_SIZE = 128
LEIDEN_RESOLUTION = 1.0
LEIDEN_SEED = 0
SOBOL_SEED = 0
MEANINGFUL_MODULE_SIZE = 10

display(pd.DataFrame(PARAMETER_RANGES, index=["minimum", "maximum"]).T)

## Load the original representation and spatial expression

Run `01_full_pipeline.ipynb` first if `results/core/` does not contain a `core_*` artifact. PCA is intentionally excluded from this experiment; each PCA representation should receive a separate cutoff calibration and landscape search later.

In [ ]:
CORE_RUN_DIR = None  # None selects the newest results/core/core_* directory.
SMOOTHED_HUMAN_PATH = DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
SMOOTHED_MOUSE_PATH = DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"

if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError(
            "No saved core run exists. Run notebooks/01_full_pipeline.ipynb first."
        )
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
for path in (SMOOTHED_MOUSE_PATH, SMOOTHED_HUMAN_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing smoothed expression input: {path}")

core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
smoothed_mouse = ad.read_h5ad(SMOOTHED_MOUSE_PATH)
smoothed_human = ad.read_h5ad(SMOOTHED_HUMAN_PATH)
display(pd.Series(core["manifest"], name="value"))

## Run or resume the search

The output directory is intentionally stable. Re-running this cell skips completed settings. The neighbor cache is also persisted separately and reused across restarts. Compact tables are rebuilt after each 128-run stage.

In [ ]:
RUN_SEARCH = True  # The experiment is resumable; set False only to inspect saved results.
EXPERIMENT_DIR = RESULTS_ROOT / "optimization/wide_sobol_no_pca_512"
CACHE_DIR = RESULTS_ROOT / "cache/original_second_order_neighbors_k500"

if RUN_SEARCH:
    search = run_wide_sobol_search(
        prof,
        C,
        expression_by_species={
            "mouse": smoothed_mouse,
            "human": smoothed_human,
        },
        output_dir=EXPERIMENT_DIR,
        cache_dir=CACHE_DIR,
        parameter_ranges=PARAMETER_RANGES,
        n_settings=N_SETTINGS,
        stage_size=STAGE_SIZE,
        resolution=LEIDEN_RESOLUTION,
        leiden_seed=LEIDEN_SEED,
        sobol_seed=SOBOL_SEED,
        meaningful_module_size=MEANINGFUL_MODULE_SIZE,
        show_heatmap=True,
    )
    display(search["summary"].sort_values("final_score", ascending=False).head(20))
else:
    print("RUN_SEARCH is False; saved results can still be inspected below.")

## Reload and inspect saved results

`raw_metrics.csv` contains the unmodified values of all 65 indicators. The heatmap uses a display-only Q10–Q90 normalization independently within each row; it does not replace or alter the raw table.

In [ ]:
if EXPERIMENT_DIR.exists():
    saved = collect_search_results(EXPERIMENT_DIR)
    display(saved["summary"].sort_values("final_score", ascending=False).head(20))
    display(saved["raw_metrics"].head())
    if not saved["raw_metrics"].empty:
        design = pd.read_csv(EXPERIMENT_DIR / "sobol_design.csv").set_index("sobol_id")
        plot_metric_heatmap(
            saved["raw_metrics"], design,
            EXPERIMENT_DIR / "key_indicator_heatmap.png", show=True,
        )
else:
    print("No saved experiment exists yet.")